# 02. Train VietLPR_v1 và VietLPR_v2

Chọn kernel `.venv-ocr`. Model dùng đúng luồng nguồn đã chốt; train trên GPU.
Paddle dùng pretrained và được chuẩn bị ở notebook 03. Mặc định chỉ kiểm tra cấu hình/model đã có.
Xem `docs/ocr.md` về loss, EMA, augment và tiêu chí chọn best.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "ocr/config.py").is_file() and (p / "configs/ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook trong thư mục dự án vn-lpr-pipeline.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PYTHON = ROOT / ".venv-ocr/Scripts/python.exe"
if Path(sys.executable).resolve() != PYTHON.resolve():
    raise RuntimeError(f"Chọn kernel .venv-ocr: {PYTHON}")
from ocr.config import load_config
from ocr.notebook import run, require_files
print("Workspace:", ROOT)

Workspace: D:\DUT\NCKH\dataset


## 1. Chọn tác vụ

Bật `RUN_TRAINING` để train/resume; bật `RUN_EXPORT` nếu chỉ cần xuất lại best sang ONNX.
`RUN_SMOKE` kiểm thử bốn mẫu trong thư mục tạm, không ghi vào trọng số thành phẩm.

In [2]:
RUN_TRAINING = False
RUN_EXPORT = False
RUN_SMOKE = False
DEVICE = "cuda"
MODELS = ("vietlpr_v1", "vietlpr_v2")

if RUN_TRAINING or RUN_SMOKE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Torch chưa nhận GPU CUDA; kiểm tra requirements/ocr-torch.txt.")
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Kiểm source, cấu hình và dữ liệu

In [3]:
from ocr.data import read_split

for source_name, marker in (
    ("VietLPR_v1", "third_party/VietLPR_v1/vietlpr/models.py"),
    ("VietLPR_v2", "third_party/VietLPR_v2/source.py"),
):
    if not (ROOT / marker).is_file():
        run(PYTHON, "-m", "ocr.bootstrap", "--source", source_name)
for name in MODELS:
    cfg = load_config(name)
    print(name, cfg["model"], "| weights:", cfg["run_dir"])
for split in ("train", "val", "test"):
    print(split, {task: len(read_split(ROOT / "vn-plate-dataset_v2", split, task)) for task in ("det", "rec")})

vietlpr_v1 {'det_size': 320, 'det_alpha': 0.5, 'rec_alpha': 0.9, 'rec_hidden': 80, 'rec_h': 32, 'rec_w': 128, 'rec_resize': 'pad', 'rec_dropout': 0.1} | weights: D:\DUT\NCKH\dataset\runs\vietlpr_v1
vietlpr_v2 {'det_size': 128, 'det_width': 1.0, 'det_fpn': 48, 'rec_width': 1.0, 'rec_dim': 128, 'rec_neck': 'attn', 'rec_h': 32, 'rec_w': 128, 'rec_resize': 'stretch', 'rec_neck_layers': 1} | weights: D:\DUT\NCKH\dataset\runs\vietlpr_v2
train {'det': 864, 'rec': 1402}
val {'det': 222, 'rec': 371}
test {'det': 124, 'rec': 190}


## 3. Smoke tuỳ chọn

In [4]:
if RUN_SMOKE:
    run(PYTHON, "-m", "ocr.smoke", "--device", DEVICE, "--full-size")
else:
    print("Smoke chưa bật.")

Smoke chưa bật.


## 4. Train/val theo từng nguồn

Thứ tự V1 DET → REC → V2 DET → REC. Cùng cấu hình/dữ liệu thì resume; thay dữ liệu sẽ lưu lượt cũ vào `runs/archive/`.
V1 giữ lịch warm-up/cosine và tiêu chí nguồn; V2 giữ train/EMA, jitter và crop từ DET dự đoán trên train.
Không tạo crop train từ val/test. Log được lưu trong `runs/notebook_logs/`.

In [5]:
from ocr.notebook import train_vietlpr

if RUN_TRAINING:
    for name in MODELS:
        train_vietlpr(PYTHON, name, device=DEVICE)
else:
    print("RUN_TRAINING=False; đọc model hiện có.")

RUN_TRAINING=False; đọc model hiện có.


## 5. Xuất best sang ONNX FP32

Xuất best theo nguồn; V2 dùng best EMA, gộp RepConv và hỗ trợ batch các dòng.
Kiểm Torch/ONNX trên val. Kết quả nằm ở `runs/<model>/onnx/`.

In [6]:
if RUN_EXPORT or RUN_TRAINING:
    for name in MODELS:
        directory = Path(load_config(name)["run_dir"])
        require_files([directory / f"{task}_best.pth" for task in ("det", "rec")], "Chưa có best checkpoint.")
        run(PYTHON, "-m", "ocr.export", "--model", name)
for name in MODELS:
    directory = Path(load_config(name)["run_dir"]) / "onnx"
    print(name, {task: (directory / f"{task}.onnx").is_file() for task in ("det", "rec")}, directory)

vietlpr_v1 {'det': True, 'rec': True} D:\DUT\NCKH\dataset\runs\vietlpr_v1\onnx
vietlpr_v2 {'det': True, 'rec': True} D:\DUT\NCKH\dataset\runs\vietlpr_v2\onnx


Sang **03_evaluate_models.ipynb** để chuẩn bị Paddle, chốt val, đo CPU và xem bảng xếp hạng.